# PES-KV: Empirical test of Persistence-Economics Selection for KV-cache eviction

Colab runner (T4 GPU). Tests the paper's Theorem 1 margin rule against published
eviction baselines (H2O, StreamingLLM, sliding window, random) on long-context
retrieval tasks. **Select GPU runtime (T4) before running.**

Cells: (1) clone + install, (2) unit tests, (3) smoke run, (4) Phase 3 sweep
(resumable), (5) Phase 5 boundary mapping, (6) stats + plots.
All results checkpoint to JSONL after every record; re-running resumes.


In [ ]:
import subprocess
REPO = "/content/persistence-routed-learning"
subprocess.run(["git", "clone", "https://github.com/Elcodwa/persistence-routed-learning.git", REPO], check=False)
%cd {REPO}
!pip install -q -e kv_pes/ pytest


In [ ]:
# Phase 1+2 gate: unit tests (eviction logic, cache, tasks, stats) -- must pass
!python -m pytest kv_pes/tests/ -q


In [ ]:
# Smoke run: 1 seed, 2 cases, one budget, GPT-2 (~2 min on T4)
!python kv_pes/scripts/run_sweep.py --model gpt2 --seeds 0 --n-cases 2 \
    --budgets 128 --tasks needle_mid --results kv_pes/results/smoke.jsonl


## Phase 3 sweep (resumable across Colab sessions)
GPT-2 full grid first (fastest); then TinyLlama and Qwen2.5 subsets.
If Colab disconnects, re-run cells 1-2 and this cell: finished records are skipped.


In [ ]:
!python kv_pes/scripts/run_sweep.py --model gpt2 --seeds 0 1 2 --n-cases 12 \
    --budgets 64 128 256 512 \
    --tasks needle_early needle_mid needle_late multifact \
    --results kv_pes/results/results.jsonl


In [ ]:
# Larger models (subset grid to fit T4 time; 2 budgets = tightness contrast)
for m in ["qwen2.5-1.5b", "tinyllama"]:
    !python kv_pes/scripts/run_sweep.py --model $m --seeds 0 1 2 --n-cases 12 \
        --budgets 128 512 \
        --tasks needle_early needle_mid needle_late multifact \
        --results kv_pes/results/results.jsonl


## Phase 5: boundary mapping -- deliberately try to break PES
Three knobs: distractor difficulty (delayed_needle), cache tightness, and
PES hyperparameters (hold_cost mu, EMA alpha). Sec. 7.4 of the paper predicts
the margin rule degrades under nonlinear credit assignment; we search for that.


In [ ]:
# distractor difficulty sweep at a tight budget
for d in [0, 4, 8, 16]:
    !python kv_pes/scripts/run_sweep.py --model gpt2 --seeds 0 1 2 --n-cases 12 \
        --budgets 128 --tasks delayed_needle --difficulty $d \
        --results kv_pes/results/boundary.jsonl


In [ ]:
# PES hyperparameter sweep on the hardest setting (its own failure surface)
!python kv_pes/scripts/run_sweep.py --model gpt2 --seeds 0 1 2 --n-cases 12 \
    --budgets 128 --tasks delayed_needle needle_mid --difficulty 8 \
    --policies pes --pes-mu 0.0 0.001 0.01 0.05 --pes-alpha 0.05 0.1 0.5 \
    --results kv_pes/results/boundary.jsonl


In [ ]:
# PES-knob sweep on larger model (does the boundary move with depth?)
!python kv_pes/scripts/run_sweep.py --model qwen2.5-1.5b --seeds 0 1 2 --n-cases 12 \
    --budgets 128 --tasks delayed_needle --difficulty 8 \
    --policies pes h2o streaming window --pes-mu 0.0 0.01 \
    --results kv_pes/results/boundary.jsonl


## Phase 4+6: paired statistics and plots


In [ ]:
import sys, json
sys.path.insert(0, "kv_pes/src")
from kv_pes.analysis import load_records, paired_vs_baselines, dose_response, needle_position_curve
from kv_pes.plots import plot_dose_response, plot_needle_position

records = load_records("kv_pes/results/results.jsonl")
print(f"{len(records)} records")

# paired PES vs every baseline, per model/budget/task -- ALL comparisons reported
rows = []
models = sorted({r["model"] for r in records})
for m in models:
    budgets = sorted({r["budget"] for r in records if r["model"] == m and r["budget"] >= 0})
    for b in budgets:
        for t in sorted({r["task"] for r in records if r["model"] == m and r["budget"] == b}):
            for cmp in paired_vs_baselines(records, model=m, task=t, budget=b):
                cmp["budget"] = b
                rows.append(cmp)

with open("kv_pes/results/paired_comparisons.json", "w") as f:
    json.dump(rows, f, indent=1)

cols = ["model","task","budget","policy_b","n","mean_a","mean_b","mean_diff","sign_test_p","a_wins","b_wins","ties"]
widths = [max(len(str(c)), *(len(str(r[c])) for r in rows)) for c in cols] if rows else []
print(" ".join(str(c).ljust(w) for c, w in zip(cols, widths)))
for r in rows:
    print(" ".join(str(round(r[c], 4) if isinstance(r[c], float) else r[c]).ljust(w) for c, w in zip(cols, widths)))


In [ ]:
# dose-response and needle-position plots
for m in models:
    for t in ["needle_mid", "multifact"]:
        plot_dose_response(dose_response(records, m, t), m, t, "kv_pes/results/figures")
    for b in [128, 512]:
        plot_needle_position(needle_position_curve(records, m, b), m, b, "kv_pes/results/figures")


In [ ]:
# download results for committing back to the repo
from google.colab import files
import shutil
shutil.make_archive("pes_kv_results", "zip", "kv_pes/results")
files.download("pes_kv_results.zip")
